# Установка зависимостей и проверка работы моделей(выполнять только один раз)

In [1]:
!pip install -q transformers datasets

In [2]:
import os
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from google.colab import drive
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed
)
from transformers.trainer_utils import get_last_checkpoint

os.environ["SAFETENSORS_FAST_GPU"] = "0"

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def set_seed(seed):
   random.seed(seed)
   np.random.seed(seed)
   torch.manual_seed(seed)
   torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)

device: cuda


In [4]:
from google.colab import drive

drive.mount('/content/drive')
drive_root = '/content/drive/MyDrive/papadyk-collab/vkr'

Mounted at /content/drive


In [5]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(HF_TOKEN)

In [6]:
import gc
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import balanced_accuracy_score, f1_score

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,  # можно оставить, если хочешь сравнивать
    BertModel,
    BertConfig,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

MODEL_NAME = "DeepPavlov/rubert-base-cased"

MAX_LENGTH = 512
BATCH_SIZE = 8
NUM_EPOCHS = 15
LR = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05

TEXT_COL = "text"
LABEL_COL = "label"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def compute_metrics(eval_pred):
    logits, labels_true = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "balanced_accuracy": balanced_accuracy_score(labels_true, preds),
        "f1_macro": f1_score(labels_true, preds, average="macro", zero_division=0),
    }

def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH
    )

class MeanPoolBertClassifier(nn.Module):
    """
    RuBERT + mean pooling по токенам (с учётом attention_mask) + линейная голова.
    """

    def __init__(self, model_name, num_labels, id2label=None, label2id=None):
        super().__init__()
        self.config = BertConfig.from_pretrained(
            model_name,
            num_labels=num_labels,
            id2label=id2label,
            label2id=label2id,
            output_hidden_states=False
        )
        self.bert = BertModel.from_pretrained(model_name, config=self.config)
        hidden_size = self.config.hidden_size
        self.dropout = nn.Dropout(self.config.hidden_dropout_prob)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def mean_pool(self, last_hidden_state, attention_mask):
        # last_hidden_state: [batch, seq, hidden]
        # attention_mask:    [batch, seq]
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)  # [batch, seq, 1]
        masked = last_hidden_state * mask
        summed = masked.sum(dim=1)               # [batch, hidden]
        counts = mask.sum(dim=1).clamp(min=1e-9) # [batch, 1]
        return summed / counts

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        labels=None,
        **kwargs
    ):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True,
        )
        last_hidden_state = outputs.last_hidden_state  # [batch, seq, hidden]
        pooled = self.mean_pool(last_hidden_state, attention_mask)  # [batch, hidden]
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.config.num_labels), labels.view(-1))

        if not self.training:
            return logits if loss is None else (loss, logits)

        return (loss, logits) if loss is not None else logits

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
TRAIN_FILE = os.path.join(drive_root, 'train.csv')
TEST_FILE = os.path.join(drive_root, 'test.csv')

### ===> очистка памяти между прогонами
for var_name in [
    "trainer",
    "train_df",
    "test_df",
    "dataset",
    "data_collator",
    "eval_metrics",
    "labels",
    "label2id",
    "id2label",
]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
### <=== очистка памяти между прогонами

train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

train_df = train_df[[TEXT_COL, LABEL_COL]].copy().dropna()
test_df = test_df[[TEXT_COL, LABEL_COL]].copy().dropna()

train_df[TEXT_COL] = train_df[TEXT_COL].astype(str).str.strip()
train_df[LABEL_COL] = train_df[LABEL_COL].astype(str).str.strip()

test_df[TEXT_COL] = test_df[TEXT_COL].astype(str).str.strip()
test_df[LABEL_COL] = test_df[LABEL_COL].astype(str).str.strip()

train_df = train_df[(train_df[TEXT_COL] != "") & (train_df[LABEL_COL] != "")].reset_index(drop=True)
test_df = test_df[(test_df[TEXT_COL] != "") & (test_df[LABEL_COL] != "")].reset_index(drop=True)

labels = sorted(train_df[LABEL_COL].unique().tolist())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

unknown_test_labels = sorted(set(test_df[LABEL_COL].unique()) - set(labels))
if unknown_test_labels:
    raise ValueError(f"В TEST_FILE есть unseen labels: {unknown_test_labels}")

train_df["label_id"] = train_df[LABEL_COL].map(label2id)
test_df["label_id"] = test_df[LABEL_COL].map(label2id)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[[TEXT_COL, "label_id"]], preserve_index=False),
    "validation": Dataset.from_pandas(test_df[[TEXT_COL, "label_id"]], preserve_index=False)
})

dataset = dataset.map(tokenize_batch, batched=True)
dataset = dataset.rename_column("label_id", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def model_init():
    return MeanPoolBertClassifier(
        model_name=MODEL_NAME,
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id,
)

training_args = TrainingArguments(
    output_dir="/tmp/hf_trainer_no_save",
    save_strategy="no",
    eval_strategy="no",
    logging_strategy="no",
    report_to="none",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    disable_tqdm=False,
)

trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
eval_metrics = trainer.evaluate(dataset["validation"])

print("\nFINAL METRICS")
print(f"balanced_accuracy: {eval_metrics['eval_balanced_accuracy']:.6f}")
print(f"f1_macro:          {eval_metrics['eval_f1_macro']:.6f}")

Map:   0%|          | 0/1419 [00:00<?, ? examples/s]

Map:   0%|          | 0/355 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
